# CareLoop Clinics — No-Show Guard
## Part 1: Ingest, validate, clean, enrich (Tasks 1.1–1.6)
Run from the repository root (the folder that contains `data/`). Outputs are written to the repository root.

## Task 1.1 — Read the file with the `csv` module only

In [29]:
import csv, json

RAW = "appointments.csv"

def parse_row(row):
    """Convert age, fee_inr, prior_no_shows to int and distance_km to float."""
    out = dict(row)
    out["age"] = int(row["age"])
    out["fee_inr"] = int(row["fee_inr"])
    out["prior_no_shows"] = int(row["prior_no_shows"])
    out["distance_km"] = float(row["distance_km"])
    return out

accepted, rejected = [], []
with open(RAW, newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    for row in reader:
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

print("Rows read:", len(accepted) + len(rejected))
print("Accepted:", len(accepted), "| Rejected:", len(rejected))
print("Rejected with blank age:", sum(r["age"].strip() == "" for r in rejected))
print("Rejected with blank distance_km:", sum(r["distance_km"].strip() == "" for r in rejected))
print("Accepted rows with impossible age (<1 or >100):", sum(r["age"] < 1 or r["age"] > 100 for r in accepted))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988 | Rejected: 32
Rejected with blank age: 12
Rejected with blank distance_km: 20
Accepted rows with impossible age (<1 or >100): 5


## Task 1.2 — Inspect and clean, pass 1

In [30]:
import pandas as pd
import numpy as np

df = pd.read_csv(RAW)
print("Shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicated rows:", df.duplicated().sum())
print("Distinct raw booking_channel spellings:", df["booking_channel"].nunique())
print(sorted(df["booking_channel"].dropna().unique()))

print("Start:", len(df))

# (a) drop exact duplicates
df = df.drop_duplicates()
print("(a) drop duplicates:", len(df))

# (b) clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("booking_channel after cleaning:", sorted(df["booking_channel"].unique()))

# (c) dates
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("(c) date conversion:", len(df), "| unreadable appt_date:", df["appt_date"].isna().sum())

# (d) drop missing appt_date or status
df = df[df["appt_date"].notna() & df["status"].notna()].copy()
print("(d) drop missing appt_date / status:", len(df))

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0
booked_o

## Task 1.3 — Clean, pass 2: validity, gaps, outliers

In [31]:
# (a) impossible ages -> missing
bad_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", bad_age.sum())
df.loc[bad_age, "age"] = np.nan
print("Missing ages in total:", df["age"].isna().sum())

# (b) fill gaps with the median
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print("Median age:", age_median, "| median distance:", dist_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) IQR fences, capped with np.clip (computed after the gaps are filled)
for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_capped = ((df[col] < low) | (df[col] > high)).sum()
    df[col] = np.clip(df[col], low, high)
    print(f"{col}: Q1={q1}, Q3={q3}, lower fence={low}, upper fence={high}, capped={n_capped}")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee after capping:", df["fee_inr"].max())

Invalid ages: 5
Missing ages in total: 17
Median age: 48.0 | median distance: 8.25
distance_km: Q1=4.625, Q3=13.0, lower fence=-7.9375, upper fence=25.5625, capped=26
fee_inr: Q1=500.0, Q3=787.5, lower fence=68.75, upper fence=1218.75, capped=3
Largest fee after capping: 1219


## Task 1.4 — Features and a NumPy summary

In [32]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({100*df['no_show'].mean():.1f}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(X.mean(axis=0), 2))

# (b) lead_days > 14
mask = X[:, 3] > 14
print(f"lead_days > 14: {mask.sum()} rows ({100*mask.mean():.1f}%)")

# (c) z-scores of distance_km (population std, np.std default)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

lead_days range: 1 to 45
No-shows: 218 of 990 (22.0%)
Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
lead_days > 14: 296 rows (29.9%)
Rows with |z| > 2 for distance_km: 53


## Task 1.5 — Enrich the clinics from the Open-Meteo API

In [33]:
import requests
clinics = pd.read_csv("clinics.csv")
temps = []
for _, clinic in clinics.iterrows():
    temp, reason = None, "Unknown error"
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={"latitude": clinic["latitude"], "longitude": clinic["longitude"],
                        "hourly": "temperature_2m", "forecast_days": 1},
                timeout=10
            )
            if response.status_code != 200:
                reason = f"HTTP {response.status_code}"
                continue
            payload = response.json()
            hourly = payload.get("hourly", {}).get("temperature_2m")
            if not hourly:
                reason = "hourly.temperature_2m missing or empty"
                continue
            temp, reason = hourly[0], None
            break
        except requests.exceptions.RequestException as exc:
            reason = str(exc)
    temps.append(temp)
    if temp is None:
        print(f'{clinic["clinic_name"]}: {reason}')
clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    19.2  
1    19.5  
2    18.6  
3    14.4  
4    14.4  
5    27.9  
6    16.6  
7    14.0  


## Task 1.6 — Export the clean table and write `load_data.sql`

In [34]:
cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
        "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
        "prior_no_shows", "status", "lead_days", "no_show"]

out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[cols]

print("appointment_id unique:", out["appointment_id"].is_unique, "| distinct:", out["appointment_id"].nunique())
print("Missing values per column (should all be 0):")
print(out.isnull().sum().to_string())
assert out["appointment_id"].is_unique and out.isnull().sum().sum() == 0

out.to_csv("appointments_clean.csv", index=False)
print("appointments_clean.csv:", out.shape)

def sql_val(v):
    """Text in single quotes, numbers unquoted, missing as NULL."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NULL"
    if isinstance(v, str):
        return "'" + v + "'"
    return str(v)

clin = pd.read_csv("clinics_enriched.csv")
n = 0
with open("load_data.sql", "w") as f:
    for _, r in clin.iterrows():          # clinics first (foreign key)
        vals = ",".join(sql_val(r[c]) for c in ["clinic_id", "clinic_name", "city", "tier", "latitude", "longitude", "temp_c"])
        f.write(f"INSERT INTO clinics VALUES ({vals});\n"); n += 1
    for _, r in out.iterrows():
        vals = ",".join(sql_val(r[c].item() if hasattr(r[c], "item") else r[c]) for c in cols)
        f.write(f"INSERT INTO appointments VALUES ({vals});\n"); n += 1
print("INSERT statements written:", n)

appointment_id unique: True | distinct: 990
Missing values per column (should all be 0):
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
appointments_clean.csv: (990, 16)
INSERT statements written: 998
